# Reranking on a Colab GPU

Runs `api/rerank.py` from the GitHub repository on a free Colab GPU, which is much faster than the laptop's CPU.
Same code, same pinned models; each run's `.json` records the git commit and the device (`cuda`), and the scores
go to their own cache file (`..._cuda.tsv`).

1. Runtime > Change runtime type > **T4 GPU**.
2. Check the settings in the first code cell below (what to rerank).
3. Runtime > Run all. The speed (new pairs a second) is printed every 20 claims.
4. The last cell downloads `rerank_results.zip`. Unzip it into the project folder (`licenta`).

Push first: Colab clones what is on GitHub, not what is on the laptop.

In [ ]:
# What to rerank. SAMPLE = 200: only the first 200 claims of the development sample (api/dev_sample.py, never a
# base claim of the targeted set), written to results/dev/. SAMPLE = None: all train claims, to results/runs/.
RERANKER = 'bge-m3'
SYSTEMS = ['B1', 'B3']
NS = [100, 30]          # N = 30 comes from the cache filled by N = 100
SAMPLE = 200
SPLIT = 'train'         # rerank.py refuses targeted and test until the reranker is fixed in hypotheses.md

In [ ]:
import subprocess
REPO = 'https://github.com/ariszaharia/boolean-search.git'
COMMIT = ''   # empty = the newest pushed commit on master; each run's .json records which commit ran
subprocess.run(['git', 'clone', '-q', REPO, 'repo'], check=True)
%cd repo
if COMMIT:
    subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
!git log --oneline -1

In [ ]:
# The laptop's transformers version, so that both places run the same code.
!pip install -q transformers==5.19.0
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run all again.'
print(torch.cuda.get_device_name(0))

In [ ]:
sample_flag = f'--sample {SAMPLE}' if SAMPLE else ''
for system in SYSTEMS:
    for n in NS:
        !python api/rerank.py --run results/runs/{system}_{SPLIT}.tsv --split {SPLIT} --n {n} --reranker {RERANKER} {sample_flag}

In [ ]:
out_dir = 'results/dev' if SAMPLE else 'results/runs'
!zip -q -r rerank_results.zip {out_dir}/*-rr*-{RERANKER}_{SPLIT}.* results/work/rerank_cache/{RERANKER}_*
from google.colab import files
files.download('rerank_results.zip')